In [ ]:
import json
import os
import re

def intelligent_replace_image(pdf_name, markdown_text, known_image_filename, bbox_string, page_id):
    """
    智能替换 Markdown 中的图片引用。
    （已修正和优化）
    """
    # 前提检查：如果裸文件名不在文本中，可能就没必要继续了
    if f"({known_image_filename})" not in markdown_text:
        print(f"Debug: {pdf_name} - ({known_image_filename}) not found in the text.")
        return markdown_text

    # 方案1: 优先匹配完整的 ![alt](filename) 格式
    # 模式现在处理括号，并且期望 known_image_filename 是一个裸文件名
    pattern = f"!\[.*?\]\({re.escape(known_image_filename)}\)"
    
    # 【已修正】使用 page_id 来构造新字符串
    new_string_for_tag = f"![](page{page_id}_{bbox_string}.jpg)"
    
    updated_text, num_replacements = re.subn(pattern, new_string_for_tag, markdown_text, count=1)

    if num_replacements > 0:
        # 成功替换了完整的标签，任务完成
        return updated_text
    else:
        # 方案2 (后备): 如果没找到完整标签，但文件名本身存在，则只替换文件名
        # 这通常发生在文件名不是以Markdown图片格式出现的情况下
        print(f"Debug: {pdf_name} - Full tag for '{known_image_filename}' not found. Falling back to filename replacement.")
        new_string_for_filename = f"page{page_id}_{bbox_string}.jpg"
        return markdown_text.replace(known_image_filename, new_string_for_filename, 1)


# --- 主程序部分 ---
INPUT_FILE = "MPDocBench.json" 
OUTPUT_PATH = "./markdown/chandra"
MD_PATH = "./markdown/chandra_md"

os.makedirs(MD_PATH, exist_ok=True)

with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)
    raw_data = {}
    for item in data:
        page_info = item["page_info"]
        images_list = page_info["images_list"]
        annotations_list = page_info["annotations_list"]
        image_path = page_info["image_path"]
        pdf_name = os.path.splitext(image_path)[0]
        if pdf_name not in raw_data:
            raw_data[pdf_name] = []
            page_id = 0
            for img, ann in zip(images_list, annotations_list):
                raw_data[pdf_name].append((page_id, img, ann))
                page_id += 1
        else:
            print(f"Warning: duplicate pdf_name {pdf_name} found. Skipping.")
            continue
    data = raw_data

for pdf_name in data:
    md_dir = os.path.join(OUTPUT_PATH, pdf_name)
    md_path = os.path.join(md_dir, f"{pdf_name}.md")
    json_path = os.path.join(md_dir, f"{pdf_name}_metadata.json")

    if os.path.exists(md_path) and os.path.exists(json_path):
        # 【已改进】使用 with open 读取文件
        try:
            with open(md_path, 'r', encoding='utf-8') as f:
                md_content = f.read()
            with open(json_path, 'r', encoding='utf-8') as f:
                json_data = json.load(f)
        except Exception as e:
            print(f"Error reading files for {pdf_name}: {e}")
            continue

        pages = json_data.get("pages", [])
        for page in pages:
            page_num = page.get("page_num")
            bboxs = page.get("bboxs", {})        
            for name, bbox in bboxs.items():
                bbox_string = f"{bbox[0]}_{bbox[1]}_{bbox[2]}_{bbox[3]}"
                md_content = intelligent_replace_image(
                    pdf_name, 
                    md_content, 
                    name,  # <- 只传递核心文件名
                    bbox_string, 
                    page_num + 1
                )
        
        try:
            with open(os.path.join(MD_PATH, f"{pdf_name}.md"), "w", encoding='utf-8') as f:
                f.write(md_content)
        except Exception as e:
            print(f"Error writing final markdown for {pdf_name}: {e}")

    else:
        print(f"Skipping {pdf_name}: Missing .md or _metadata.json file.")

print("Processing finished.")